# BIDS Dataset Metadata

Run this notebook once per dataset to complete the BIDS metadata that cannot be captured inside MUedit.

Covers:
1. **Dataset description** — authors, license, funding, ethics approvals
2. **Task metadata** — task description and participant instructions (inherited by all recordings for that task)
3. **Recording-level details** — skin preparation, institution, ground electrode, coordinate system description
4. **Validation** — run `bids-validator` and display the result

> All patch operations are idempotent: re-running the cell updates existing values without touching unrelated fields.

## 0. Setup

In [ ]:
import json
from pathlib import Path
from muedit.api.config import resolve_bids_root

# ── Set your project name (folder name inside data/) ─────────────────────────
PROJECT = "myStudy"
# ─────────────────────────────────────────────────────────────────────────────

BIDS_ROOT = resolve_bids_root(PROJECT)
assert BIDS_ROOT.exists(), f"BIDS root not found: {BIDS_ROOT}"
print("BIDS root:", BIDS_ROOT)


def _patch_json(path: Path, updates: dict) -> None:
    """Update existing JSON file with new key-value pairs (non-destructive)."""
    data = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    data.update(updates)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"  updated: {path.relative_to(BIDS_ROOT)}")

### Inspect current state

Shows what MUedit has already written so you know what each section below still needs to fill.

In [ ]:
desc_path = BIDS_ROOT / "dataset_description.json"
if desc_path.exists():
    print("\u2500\u2500 dataset_description.json \u2500\u2500")
    print(json.dumps(json.loads(desc_path.read_text(encoding="utf-8")), indent=2))

print("\n\u2500\u2500 First *_emg.json sidecar found \u2500\u2500")
for p in sorted(BIDS_ROOT.rglob("*_emg.json")):
    print(json.dumps(json.loads(p.read_text(encoding="utf-8")), indent=2))
    break

print("\n\u2500\u2500 First *_coordsystem.json found \u2500\u2500")
for p in sorted(BIDS_ROOT.rglob("*_coordsystem.json")):
    print(json.dumps(json.loads(p.read_text(encoding="utf-8")), indent=2))
    break

## 1. Dataset Description

MUedit creates `dataset_description.json` with `Name`, `BIDSVersion`, and `DatasetType` already set.  
This cell fills in the fields left as placeholders (`Authors: []`, `License: "n/a"`) plus the optional provenance fields.

**Example** (MUniverse — Avrillon et al., 2024):
```python
AUTHORS = ["Avrillon, Simon", "Hug, Francois", "Enoka, Roger M.", "Caillet, Arnault H.", "Farina, Dario"]
LICENSE = "CC0 BY 4.0"
FUNDING = ["Agence Nationale de la Recherche (ANR-16-IDEX-0007)", "European Research Council (#810346)"]
ETHICS_APPROVALS = ["Imperial College London (Study 18IC4685)",
                    "Comite de Protection des Personnes Ouest III (Study 23.00453.000166)"]
REFERENCES_AND_LINKS = ["https://doi.org/10.7554/eLife.97085.3"]
```

In [ ]:
# ── Edit the values below ─────────────────────────────────────────────────────────────────────────────
AUTHORS = [
    "Last, First",          # e.g. "Avrillon, Simon"
    # add more authors ...
]
LICENSE              = "CC0"                              # e.g. "CC0", "CC-BY-4.0"
FUNDING              = ["Grant agency, award number"]     # e.g. "ERC (#810346)"
ETHICS_APPROVALS     = ["Institution, approval number"]   # e.g. "Imperial College London (Study 18IC4685)"
ACKNOWLEDGEMENTS     = ""
HOW_TO_ACKNOWLEDGE   = ""   # citation instruction
REFERENCES_AND_LINKS = []   # DOIs / URLs
# ─────────────────────────────────────────────────────────────────────────────

desc_path = BIDS_ROOT / "dataset_description.json"
updates = {
    "Authors": AUTHORS,
    "License": LICENSE,
    "Funding": FUNDING,
    "EthicsApprovals": ETHICS_APPROVALS,
}
if ACKNOWLEDGEMENTS:
    updates["Acknowledgements"] = ACKNOWLEDGEMENTS
if HOW_TO_ACKNOWLEDGE:
    updates["HowToAcknowledge"] = HOW_TO_ACKNOWLEDGE
if REFERENCES_AND_LINKS:
    updates["ReferencesAndLinks"] = REFERENCES_AND_LINKS
_patch_json(desc_path, updates)

print(json.dumps(json.loads(desc_path.read_text(encoding="utf-8")), indent=2))

## 2. Task Metadata

Writes (or updates) a `task-<label>_emg.json` file at the dataset root.  
BIDS inheritance applies: every recording with that task label inherits `TaskDescription` and `Instructions` from here.

> **Already in the app?**  
> If you filled `TaskDescription` in MUedit’s save dialog, it is already written into each `*_emg.json` sidecar.  
> Run the inspection cell above to check. Creating the root-level task file is cleaner (avoids duplication across sidecars) but not required.

**Example** (MUniverse — `isometric10percentmvc` task):
```python
TASK_LABEL       = "isometric10percentmvc"
TASK_DESCRIPTION = "Trapezoidal contraction: MVC level: 10 % MVC; MVC rate during ramps: 5 % MVC / s; plateau duration: 20 s."
INSTRUCTIONS     = "Follow path provided via visual feedback."
```

In [ ]:
# ── Edit the values below ─────────────────────────────────────────────────────────────────────────────
TASK_LABEL       = "trapezoid"          # must match task-<label> used during recording
                                         # e.g. "isometric10percentmvc"
TASK_DESCRIPTION = (
    "Isometric contraction at a target force level following a trapezoidal "
    "profile displayed on screen."
    # e.g. "Trapezoidal contraction: MVC level: 10 % MVC; ramp rate: 5 % MVC / s; plateau: 20 s."
)
INSTRUCTIONS = (
    "Track the force target displayed on the screen as accurately as possible."
    # e.g. "Follow path provided via visual feedback."
)
# ─────────────────────────────────────────────────────────────────────────────

task_path = BIDS_ROOT / f"task-{TASK_LABEL}_emg.json"
_patch_json(task_path, {
    "TaskName": TASK_LABEL,
    "TaskDescription": TASK_DESCRIPTION,
    "Instructions": INSTRUCTIONS,
})

print(json.dumps(json.loads(task_path.read_text(encoding="utf-8")), indent=2))

## 3. Recording-Level Details

Patches each `*_emg.json` sidecar and all `*_coordsystem.json` files with acquisition details that are the same across recordings.

> **Already set by MUedit** (no need to repeat here):  
> `Manufacturer`, `ManufacturersModelName`, `Gain`, `HardwareFilters`, `SamplingFrequency`,  
> `SoftwareVersions`, `PowerLineFrequency`, `RecordingType`, `SoftwareFilters`,  
> `EMGPlacementScheme`, `EMGPlacementSchemeDescription`, `TaskName`.  
> The coordinate system files already have `EMGCoordinateSystem`, `EMGCoordinateUnits`, and a generic `EMGCoordinateSystemDescription` — refine the description below if needed.

Set `SUBJECT_FILTER` / `SESSION_FILTER` to `None` to apply to all subjects/sessions.

**Example** (MUniverse):
```python
SKIN_PREPARATION    = "The skin was shaved and cleaned with an abrasive pad and water."
EMG_GROUND          = "R2 — two bands damped with water placed around the ankle"
INSTITUTION_NAME    = "Imperial College London"
INSTITUTION_ADDRESS = "London W12 0BZ, United Kingdom"
INSTITUTION_DEPT    = "Department of Bioengineering"
COORD_SYSTEM_DESCRIPTION = (
    "x-axis: proximal -> distal; y-axis: medial -> lateral; "
    "all local electrode coordinates (per grid) are mapped on a single global coordinate system."
)
```

In [ ]:
# ── Edit the values below ─────────────────────────────────────────────────────────────────────────────
SUBJECT_FILTER = None    # e.g. "01" to patch one subject, or None for all
SESSION_FILTER = None    # e.g. "1"  to patch one session,  or None for all

SKIN_PREPARATION    = "Abrasion and alcohol wipe"
                      # e.g. "The skin was shaved and cleaned with an abrasive pad and water."
EMG_GROUND          = "Bony prominence of the lateral epicondyle"
                      # e.g. "R2 — two bands damped with water placed around the ankle"
INSTITUTION_NAME    = "My University"
                      # e.g. "Imperial College London"
INSTITUTION_ADDRESS = "City, Country"
                      # e.g. "London W12 0BZ, United Kingdom"
INSTITUTION_DEPT    = ""   # e.g. "Department of Bioengineering" (leave empty to skip)

# Refine the coordinate system description written by MUedit.
# Leave empty to keep the existing value.
COORD_SYSTEM_DESCRIPTION = ""
# e.g.: "x-axis: proximal -> distal; y-axis: medial -> lateral; "
#        "all local electrode coordinates mapped on a single global coordinate system."
# ─────────────────────────────────────────────────────────────────────────────

emg_json_updates = {k: v for k, v in {
    "SkinPreparation": SKIN_PREPARATION,
    "EMGGround": EMG_GROUND,
    "InstitutionName": INSTITUTION_NAME,
    "InstitutionAddress": INSTITUTION_ADDRESS,
    "InstitutionalDepartmentName": INSTITUTION_DEPT,
}.items() if v}

patched = 0
for emg_json_path in sorted(BIDS_ROOT.rglob("*_emg.json")):
    parts = emg_json_path.parts
    sub = next((p for p in parts if p.startswith("sub-")), None)
    ses = next((p for p in parts if p.startswith("ses-")), None)
    if SUBJECT_FILTER and sub != f"sub-{SUBJECT_FILTER}":
        continue
    if SESSION_FILTER and ses != f"ses-{SESSION_FILTER}":
        continue
    _patch_json(emg_json_path, emg_json_updates)
    patched += 1

if COORD_SYSTEM_DESCRIPTION:
    for cs_path in sorted(BIDS_ROOT.rglob("*_coordsystem.json")):
        _patch_json(cs_path, {"EMGCoordinateSystemDescription": COORD_SYSTEM_DESCRIPTION})

print(f"\nPatched {patched} *_emg.json file(s).")

## 4. Validation

Runs the schema-based BIDS validator against the dataset and prints the result.

The validator is the official BIDS tool, distributed as a Deno module
(`jsr:@bids/validator`). It is the only validator that supports the EMG BEP, so
it is the one to use here. `deno run` fetches and caches the validator on first
use — nothing else to install.

### Installing Deno

`deno` is already pinned in `environment.yml`, so if you created the MUedit
conda environment it is on your PATH and the cell below just works. If `deno` is
missing, install it one of these ways and re-run the cell:

- **conda (recommended — matches the MUedit env):**
  ```bash
  conda install -c conda-forge deno
  ```
- **official installer (macOS / Linux):**
  ```bash
  curl -fsSL https://deno.land/install.sh | sh
  ```
- **official installer (Windows PowerShell):**
  ```powershell
  irm https://deno.land/install.ps1 | iex
  ```

> The PyPI/conda `bids-validator` *package* is **not** this CLI: it is a
> filename-checking library whose bundled rules cover `eeg/ieeg/meg/nirs` but
> **not `emg`**, so it would reject MUedit's `*_emg.*` files. Use the Deno
> validator below instead.
>
> EMG is still a BIDS Extension Proposal, so a few warnings for `*_emg.*` files
> are expected on validator versions that predate full EMG support. The
> `derivatives/` outputs are excluded from validation via `.bidsignore`.

In [ ]:
import shutil
import subprocess

# The official BIDS validator is a Deno module (jsr:@bids/validator) — the only
# validator that supports the EMG BEP. `deno run` fetches it on first use.
if not shutil.which("deno"):
    raise RuntimeError(
        "Deno not found. Install it, then re-run this cell:\n"
        "  • conda (matches the MUedit env):  conda install -c conda-forge deno\n"
        "  • official installer (macOS/Linux): curl -fsSL https://deno.land/install.sh | sh\n"
        "  • official installer (Windows):     irm https://deno.land/install.ps1 | iex"
    )

cmd = ["deno", "run", "-ERN", "jsr:@bids/validator", str(BIDS_ROOT)]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout or result.stderr)